In [ ]:
# Librerie di sistema e utilità
import os
import warnings
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# Librerie per elaborazione audio
import librosa

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torchvision import transforms
import timm

# Ignoriamo i warning
warnings.filterwarnings("ignore")

print("Librerie importate con successo!")
print(f"PyTorch versione: {torch.__version__}")
print(f"timm versione: {timm.__version__}")

In [ ]:
class Config:
    def __init__(self):
        # Imposta i percorsi di base in base all'ambiente
        self.COMPETITION_NAME = "birdclef-2025"
        self.BASE_DIR = f"/kaggle/input/{self.COMPETITION_NAME}"
        self.OUTPUT_DIR = "/kaggle/working"
        self.MODELS_DIR = "/kaggle/input"  # Per i modelli pre-addestrati
            
        # Imposta subito i percorsi derivati per l'ambiente Kaggle
        self._setup_derived_paths()
        
        # Parametri per il preprocessing audio
        self.SR = 32000      # Sample rate
        self.DURATION = 5    # Durata dei clip in secondi
        self.N_MELS = 224    # Numero di bande Mel
        self.N_FFT = 2048    # Dimensione finestra FFT
        self.HOP_LENGTH = 512  # Hop length per STFT
        self.FMIN = 48       # Frequenza minima per lo spettrogramma Mel
        self.FMAX = 16000    # Frequenza massima
        self.POWER = 2       # Esponente per calcolo spettrogramma
            
        # Parametri per il device
        self.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
        
        # Parametri per inference/submission
        self.TEST_CLIP_DURATION = 5  # Durata dei segmenti per la predizione (secondi)
        self.N_CLASSES = 0  # Sarà impostato dopo aver caricato i dati

    def _setup_derived_paths(self):
        """Imposta i percorsi derivati basati su BASE_DIR"""
        self.TRAIN_AUDIO_DIR = os.path.join(self.BASE_DIR, "train_audio")
        self.TEST_SOUNDSCAPES_DIR = os.path.join(self.BASE_DIR, "test_soundscapes")
        self.TRAIN_CSV_PATH = os.path.join(self.BASE_DIR, "train.csv")
        self.TAXONOMY_CSV_PATH = os.path.join(self.BASE_DIR, "taxonomy.csv") 
        self.SAMPLE_SUB_PATH = os.path.join(self.BASE_DIR, "sample_submission.csv")

# Inizializza la configurazione
config = Config()

print(f"Device utilizzato: {config.DEVICE}")
print(f"Directory Test Soundscapes: {config.TEST_SOUNDSCAPES_DIR}")
print(f"Path Sample Submission: {config.SAMPLE_SUB_PATH}")

In [ ]:
def load_metadata():
    """
    Carica e prepara i metadati dal file CSV di training.
    
    Returns:
        tuple: all_species
    """
    print(f"Caricamento metadati da: {config.TRAIN_CSV_PATH}")
    train_df = pd.read_csv(config.TRAIN_CSV_PATH)
    sample_sub_df = pd.read_csv(config.SAMPLE_SUB_PATH)
    
    # Estrai tutte le etichette uniche
    train_primary_labels = train_df['primary_label'].unique()
    train_secondary_labels = set([lbl for sublist in train_df['secondary_labels'].apply(eval) 
                                 for lbl in sublist if lbl])
    submission_species = sample_sub_df.columns[1:].tolist()  # Escludi row_id
    
    # Combina tutte le possibili etichette
    all_species = sorted(list(set(train_primary_labels) | train_secondary_labels | set(submission_species)))
    N_CLASSES = len(all_species)
    config.N_CLASSES = N_CLASSES  # Aggiorna il numero di classi nella configurazione
    
    print(f"Numero totale di specie trovate: {N_CLASSES}")
    
    return all_species

# Carica le specie
all_species = load_metadata()

In [ ]:
# Crea una singola istanza della trasformazione MelSpectrogram da riutilizzare
mel_transform = T.MelSpectrogram(
    sample_rate=config.SR,
    n_fft=config.N_FFT,
    win_length=None,
    hop_length=config.HOP_LENGTH,
    f_min=config.FMIN,
    f_max=config.FMAX,
    n_mels=config.N_MELS,
    window_fn=torch.hann_window,
    power=config.POWER,
    normalized=False,
    onesided=True,
    norm="slaney",
    mel_scale="slaney"
)

# Funzione di conversione a dB e normalizzazione
def amplitude_to_db(spectrogram):
    """Converti spettrogramma in scala dB e normalizza tra 0-1"""
    # Converti in dB
    spectrogram_db = 10.0 * torch.log10(torch.clamp(spectrogram, min=1e-10))
    
    # Normalizza
    min_val = torch.min(spectrogram_db)
    max_val = torch.max(spectrogram_db)
    if max_val > min_val:
        return (spectrogram_db - min_val) / (max_val - min_val)
    else:
        return torch.zeros_like(spectrogram_db)

In [ ]:
# Funzioni helper per l'inizializzazione e la manipolazione dei dati
def init_layer(layer):
    nn.init.xavier_uniform_(layer.weight)
    if hasattr(layer, "bias"):
        if layer.bias is not None:
            layer.bias.data.fill_(0.)

def init_bn(bn):
    bn.bias.data.fill_(0.)
    bn.weight.data.fill_(1.0)

def interpolate(x: torch.Tensor, ratio: int):
    """Interpolate data in time domain. This is used to compensate the
    resolution reduction in downsampling of a CNN.
    Args:
      x: (batch_size, time_steps, classes_num)
      ratio: int, ratio to interpolate
    Returns:
      upsampled: (batch_size, time_steps * ratio, classes_num)
    """
    (batch_size, time_steps, classes_num) = x.shape
    upsampled = x[:, :, None, :].repeat(1, 1, ratio, 1)
    upsampled = upsampled.reshape(batch_size, time_steps * ratio, classes_num)
    return upsampled

def pad_framewise_output(framewise_output: torch.Tensor, frames_num: int):
    """Pad framewise_output to the same length as input frames. The pad value
    is the same as the value of the last frame.
    Args:
      framewise_output: (batch_size, frames_num, classes_num)
      frames_num: int, number of frames to pad
    Outputs:
      output: (batch_size, frames_num, classes_num)
    """
    pad = framewise_output[:, -1:, :].repeat(
        1, frames_num - framewise_output.shape[1], 1)
    output = torch.cat((framewise_output, pad), dim=1)
    return output

In [ ]:
class AttBlockV2(nn.Module):
    def __init__(self, in_features: int, out_features: int, activation="linear"):
        super().__init__()

        self.activation = activation
        self.att = nn.Conv1d(
            in_channels=in_features,
            out_channels=out_features,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=True)
        self.cla = nn.Conv1d(
            in_channels=in_features,
            out_channels=out_features,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=True)

        self.init_weights()

    def init_weights(self):
        init_layer(self.att)
        init_layer(self.cla)

    def forward(self, x):
        # x: (n_samples, n_in, n_time)
        norm_att = torch.softmax(torch.tanh(self.att(x)), dim=-1)
        cla = self.nonlinear_transform(self.cla(x))
        x = torch.sum(norm_att * cla, dim=2)
        return x, norm_att, cla

    def nonlinear_transform(self, x):
        if self.activation == 'linear':
            return x
        elif self.activation == 'sigmoid':
            return torch.sigmoid(x)

In [ ]:
class EfficientNetSED(nn.Module):
    def __init__(self, base_model_name: str, pretrained=True, num_classes=264):
        super().__init__()
        self.interpolate_ratio = 30  # Downsampled ratio
        
        # Carica il modello EfficientNet preaddestrato da timm
        self.base_model = timm.create_model(
            base_model_name, 
            pretrained=pretrained, 
            in_chans=1,  # Per input monocromatico (spettrogramma mel)
            features_only=True
        )
        
        # Ottieni la dimensione delle feature dall'ultimo layer
        self.in_features = self.base_model.feature_info.channels()[-1]
        
        self.fc1 = nn.Linear(self.in_features, self.in_features, bias=True)
        self.att_block = AttBlockV2(self.in_features, num_classes, activation="sigmoid")

        self.init_weight()

    def init_weight(self):
        init_layer(self.fc1)

    def forward(self, input):
        # Ci aspettiamo input di dimensione (batch_size, channels, freq, frames)
        frames_num = input.size(3)
        
        # Estrai features - ottiene una lista di tensori a diverse profondità della rete
        features = self.base_model(input)
        
        # Prendi l'ultimo tensore di features (quello con più alto livello di astrazione)
        x = features[-1]
        
        # Media sulle frequenze (dim=2)
        x = torch.mean(x, dim=2)

        # Channel smoothing
        x1 = F.max_pool1d(x, kernel_size=3, stride=1, padding=1)
        x2 = F.avg_pool1d(x, kernel_size=3, stride=1, padding=1)
        x = x1 + x2

        x = F.dropout(x, p=0.5, training=self.training)
        x = x.transpose(1, 2)
        x = F.relu_(self.fc1(x))
        x = x.transpose(1, 2)
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Applicazione del blocco di attenzione
        (clipwise_output, norm_att, segmentwise_output) = self.att_block(x)
        
        # Calcolo dei vari output
        logit = torch.sum(norm_att * self.att_block.cla(x), dim=2)
        segmentwise_logit = self.att_block.cla(x).transpose(1, 2)
        segmentwise_output = segmentwise_output.transpose(1, 2)

        # Get framewise output
        framewise_output = interpolate(segmentwise_output, self.interpolate_ratio)
        framewise_output = pad_framewise_output(framewise_output, frames_num)

        framewise_logit = interpolate(segmentwise_logit, self.interpolate_ratio)
        framewise_logit = pad_framewise_output(framewise_logit, frames_num)

        output_dict = {
            "framewise_output": framewise_output,
            "segmentwise_output": segmentwise_output,
            "logit": logit,
            "framewise_logit": framewise_logit,
            "clipwise_output": clipwise_output
        }

        return output_dict

In [ ]:
def create_efficientnet_sed_model(config):
    """
    Crea un'istanza del modello EfficientNetSED con la configurazione specificata.
    """
    # Scegli la versione di EfficientNet da usare
    model_name = 'tf_efficientnet_b0.ns_jft_in1k'
    
    model = EfficientNetSED(
        base_model_name=model_name,
        pretrained=False,
        num_classes=config.N_CLASSES
    )
    
    return model

# Percorso del modello pre-addestrato (aggiorna con il tuo percorso)
model_path = "/kaggle/input/birdclef-efficientnet-sed-model/birdclef_efficientNETJFT_SED_35_Epochs_best.pth"

# Inizializza il modello
model = create_efficientnet_sed_model(config).to(config.DEVICE)

# Carica il modello pre-addestrato
try:
    print(f"Caricamento del modello SED pre-addestrato da {model_path}...")
    checkpoint = torch.load(model_path, map_location=config.DEVICE)
    
    # Verifica se è un dict con model_state_dict o direttamente state_dict
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Modello caricato con successo (epoca: {checkpoint.get('epoch', 'N/A')}, val_loss: {checkpoint.get('best_val_loss', 'N/A')})")
    else:
        model.load_state_dict(checkpoint)
        print("Modello caricato con successo")
    
    model.eval()  # Imposta il modello in modalità valutazione
except Exception as e:
    print(f"Errore nel caricamento del modello: {e}")
    raise

In [ ]:
def generate_submission_with_advanced_processing(model, device=config.DEVICE):
    """
    Genera un file di submission con tecniche avanzate di post-processing:
    1. Padding strategico per centrare i primi e ultimi segmenti
    2. Smoothing con kernel [0.1, 0.2, 0.4, 0.2, 0.1]
    3. Delta shift TTA
    
    Args:
        model: Modello PyTorch addestrato
        device: Device per inferenza ('cuda' o 'cpu')
        
    Returns:
        pd.DataFrame: DataFrame di submission
    """
    model.to(device)
    model.eval()
    
    # Set seed per riproducibilità
    np.random.seed(42)
    
    # Kernel di smoothing
    smoothing_kernel = np.array([0.1, 0.2, 0.4, 0.2, 0.1])
    
    # Delta shifts per TTA (in campioni)
    delta_shifts = [-1600, -800, 0, 800, 1600]  # ±25ms, ±50ms con SR=32000
    
    # Percorso dei test soundscapes
    test_soundscape_path = config.TEST_SOUNDSCAPES_DIR
    test_soundscapes = [os.path.join(test_soundscape_path, afile) 
                        for afile in sorted(os.listdir(test_soundscape_path)) 
                        if afile.endswith('.ogg')]
    
    print(f"Elaborazione di {len(test_soundscapes)} file soundscape...")
    
    # Crea DataFrame per le predizioni finali
    predictions = pd.DataFrame(columns=['row_id'] + all_species)
    
    for soundscape in tqdm(test_soundscapes, desc="Elaborazione soundscapes"):
        # Carica audio
        sig, rate = librosa.load(path=soundscape, sr=config.SR)
        
        # Calcola la lunghezza del segmento in campioni
        segment_length = rate * config.TEST_CLIP_DURATION
        
        # Aggiungi padding strategico all'inizio e alla fine del segnale
        # Metà della lunghezza del segmento per centrare i segmenti di inizio e fine
        padding_length = segment_length // 2
        padded_sig = np.pad(sig, (padding_length, padding_length), mode='constant')
        
        # Suddividi il segnale paddato in segmenti sovrapposti
        chunks = []
        start_indices = []
        
        for i in range(0, len(sig), segment_length):
            # Indice di inizio nel segnale paddato
            start_idx = i + padding_length
            start_indices.append(start_idx)
            
            # Estrai il segmento
            chunk = padded_sig[start_idx:start_idx + segment_length]
            
            # Padda se necessario (per l'ultimo segmento)
            if len(chunk) < segment_length:
                chunk = np.pad(chunk, (0, segment_length - len(chunk)), mode='constant')
                
            chunks.append(chunk)
        
        # File name per i row_id
        file_name = os.path.basename(soundscape).split('.')[0]
        
        # Lista per memorizzare le predizioni raw e row_ids per questo file
        raw_predictions = []
        row_ids = []
        
        # Prima fase: genera predizioni per ogni chunk con Delta Shift TTA
        for i, chunk in enumerate(chunks):
            # Calcola row_id (nota: i segmenti sono centrati)
            row_id = f"{file_name}_{i * config.TEST_CLIP_DURATION + config.TEST_CLIP_DURATION}"
            row_ids.append(row_id)
            
            # Applica Delta Shift TTA
            chunk_predictions = []
            
            for shift in delta_shifts:
                # Applica lo shift
                if shift != 0:
                    shifted_chunk = np.roll(chunk, shift)
                    # Azzera i bordi per evitare artefatti
                    if shift > 0:
                        shifted_chunk[:shift] = 0
                    else:
                        shifted_chunk[shift:] = 0
                else:
                    shifted_chunk = chunk
                
                # Calcola spettrogramma Mel
                mel_spec = librosa.feature.melspectrogram(
                    y=shifted_chunk, sr=config.SR,
                    n_fft=config.N_FFT,
                    hop_length=config.HOP_LENGTH,
                    n_mels=config.N_MELS,
                    fmin=config.FMIN,
                    fmax=config.FMAX
                )
                
                # Converti in scala logaritmica (dB) e normalizza
                log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
                min_val = np.min(log_mel_spec)
                max_val = np.max(log_mel_spec)
                if max_val > min_val:
                    log_mel_spec = (log_mel_spec - min_val) / (max_val - min_val)
                else:
                    log_mel_spec = np.zeros_like(log_mel_spec)
                
                # Prepara il tensor per il modello
                log_mel_spec = np.expand_dims(np.expand_dims(log_mel_spec, axis=0), axis=0)
                input_tensor = torch.tensor(log_mel_spec, dtype=torch.float32).to(device)
                
                # Resize a 224x224 come nel training
                resize_transform = transforms.Resize((224, 224), 
                    interpolation=transforms.InterpolationMode.BICUBIC)
                input_tensor = resize_transform(input_tensor)

                # Effettua predizione
                with torch.no_grad():
                    output_dict = model(input_tensor)
                    # Utilizza direttamente clipwise_output che è già passato per sigmoid nel modello SED
                    scores = output_dict['clipwise_output'].cpu().numpy()[0]
                
                chunk_predictions.append(scores)
            
            # Media le predizioni di tutti i delta shifts
            avg_prediction = np.mean(chunk_predictions, axis=0)
            raw_predictions.append(avg_prediction)
        
        # Seconda fase: post-processing con smoothing temporale avanzato
        smoothed_predictions = []
        n_chunks = len(raw_predictions)
        
        for i in range(n_chunks):
            # Usa il kernel di smoothing [0.1, 0.2, 0.4, 0.2, 0.1]
            weighted_sum = raw_predictions[i] * smoothing_kernel[2]  # Peso centrale (0.4)
            weight_sum = smoothing_kernel[2]
            
            # Aggiungi contributo delle due finestre precedenti se esistono
            if i > 0:
                weighted_sum += raw_predictions[i-1] * smoothing_kernel[1]  # Peso 0.2
                weight_sum += smoothing_kernel[1]
                if i > 1:
                    weighted_sum += raw_predictions[i-2] * smoothing_kernel[0]  # Peso 0.1
                    weight_sum += smoothing_kernel[0]
            
            # Aggiungi contributo delle due finestre successive se esistono
            if i < n_chunks - 1:
                weighted_sum += raw_predictions[i+1] * smoothing_kernel[3]  # Peso 0.2
                weight_sum += smoothing_kernel[3]
                if i < n_chunks - 2:
                    weighted_sum += raw_predictions[i+2] * smoothing_kernel[4]  # Peso 0.1
                    weight_sum += smoothing_kernel[4]
            
            # Normalizza per i pesi effettivamente utilizzati
            smoothed_pred = weighted_sum / weight_sum
            smoothed_predictions.append(smoothed_pred)
        
        # Terza fase: aggiungi le predizioni smoothed al DataFrame finale
        for i, (row_id, pred) in enumerate(zip(row_ids, smoothed_predictions)):
            new_row = pd.DataFrame([[row_id] + list(pred)], columns=['row_id'] + all_species)
            predictions = pd.concat([predictions, new_row], axis=0, ignore_index=True)
    
    # Salva la submission come CSV
    submission_path = os.path.join(config.OUTPUT_DIR, "submission.csv")
    predictions.to_csv(submission_path, index=False)
    print(f"Submission con post-processing avanzato salvata in: {submission_path}")
    
    return predictions

In [ ]:
# Genera submission con tecniche di post-processing avanzate
print("\nGenerazione del file di submission con post-processing avanzato...")
submission_df = generate_submission_with_advanced_processing(model)

# Mostra anteprima
print("\nAnteprima del file di submission:")
print(submission_df.head())
print(f"\nShape della submission: {submission_df.shape}")

In [ ]:
# Verifica che la submission sia corretta
sample_sub = pd.read_csv(config.SAMPLE_SUB_PATH)

# Controlla che i row_id siano gli stessi
missing_rows = set(sample_sub['row_id']) - set(submission_df['row_id'])
extra_rows = set(submission_df['row_id']) - set(sample_sub['row_id'])

if len(missing_rows) > 0:
    print(f"ATTENZIONE: {len(missing_rows)} row_id mancanti nella submission!")
else:
    print("✓ Tutti i row_id sono presenti")

if len(extra_rows) > 0:
    print(f"ATTENZIONE: {len(extra_rows)} row_id extra nella submission!")
else:
    print("✓ Nessun row_id extra")

# Controlla che tutte le colonne siano presenti
if set(all_species) == set(sample_sub.columns[1:]):
    print("✓ Tutte le colonne delle specie sono corrette")
else:
    print("ATTENZIONE: Le colonne delle specie non corrispondono!")

print("\nSubmission generata con successo!")